# TAMER OCR v2.2 — Clean Slate Training Pipeline

This notebook is a **fully self-contained, deterministic orchestration layer**. It guarantees a completely clean state before running, applies critical architectural fixes to the underlying codebase, and runs the training pipeline seamlessly on either Google Colab or Kaggle.

---

### ⏱️ Estimated Execution Times

**1. Preprocessing (Data Download, InkML Rendering, Image Normalization):**
*   **CPU Bound:** ~1.5 to 2.5 hours. (Rendering 100k+ CROHME InkML files is CPU intensive).
*   *Note: Once completed, this is pushed to your HuggingFace dataset repo. Future runs will skip this and download the preprocessed data in ~5 minutes.*

**2. Training (Swin-Base + 6-Layer Transformer Decoder | ~430k samples):**
*   **NVIDIA T4 (Kaggle / Colab Free):** ~18-22 mins per epoch. Full 150 epochs = **~45-55 hours**. *(Requires session restarts; the auto-resume will handle this).* 
*   **NVIDIA P100 (Kaggle):** ~12-15 mins per epoch. Full 150 epochs = **~30-38 hours**.
*   **NVIDIA A100 (Colab Pro):** ~4-6 mins per epoch. Full 150 epochs = **~10-15 hours**.

---
### ⚠️ Important
Do not add training logic to this notebook. All logic is modularized in the cloned repository. This notebook purely handles the environment, state, and execution.

## 1. Absolute Purge (Clean Slate Protocol)
This cell explicitly deletes all previous caches, repos, checkpoints, and outputs to prevent state corruption or feature shock from previous buggy runs.

In [ ]:
import os
import shutil
import gc

def purge_environment():
    print("☣️ INITIATING CLEAN SLATE PROTOCOL...")
    
    directories_to_nuke = [
        '/content/AI_PROJECT_TAMER_Complete',
        '/kaggle/working/AI_PROJECT_TAMER_Complete',
        '/content/data',
        '/kaggle/working/data',
        '/content/outputs',
        '/kaggle/working/outputs',
        '/content/checkpoints',
        '/kaggle/working/checkpoints',
        '/content/logs',
        '/kaggle/working/logs',
        '/root/.cache/huggingface/datasets'
    ]
    
    for directory in directories_to_nuke:
        if os.path.exists(directory):
            print(f"  [DELETING] {directory}")
            shutil.rmtree(directory, ignore_errors=True)
            
    # Verify deletion
    for directory in directories_to_nuke:
        assert not os.path.exists(directory), f"Failed to delete {directory}!"
            
    gc.collect()
    print("✅ Environment successfully purged. Ready for deterministic build.")

purge_environment()

## 2. Environment Setup & Authentication

In [ ]:
import os
import getpass

# Detect Environment
IS_KAGGLE = os.path.exists('/kaggle')
WORK_DIR = '/kaggle/working' if IS_KAGGLE else '/content'
os.chdir(WORK_DIR)

print(f"🌍 Environment: {'Kaggle' if IS_KAGGLE else 'Colab'}")
print(f"📂 Working Directory: {WORK_DIR}\n")

print("🔑 AUTHENTICATION")
# HuggingFace Token
hf_token = os.getenv('HF_TOKEN', '')
if not hf_token:
    hf_token = getpass.getpass('Enter HuggingFace Token (WRITE access needed): ')
os.environ['HF_TOKEN'] = hf_token

# Kaggle Credentials
kaggle_username = os.getenv('KAGGLE_USERNAME', '')
if not kaggle_username:
    kaggle_username = input('Enter Kaggle Username: ').strip()
    
kaggle_key = os.getenv('KAGGLE_KEY', '')
if not kaggle_key:
    kaggle_key = getpass.getpass('Enter Kaggle API Key: ')
    
os.environ['KAGGLE_USERNAME'] = kaggle_username
os.environ['KAGGLE_KEY'] = kaggle_key

# Configure Kaggle CLI Auth File
kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)
with open(os.path.join(kaggle_dir, 'kaggle.json'), 'w') as f:
    f.write(f'{{"username":"{kaggle_username}","key":"{kaggle_key}"}}')
os.chmod(os.path.join(kaggle_dir, 'kaggle.json'), 0o600)
print("✅ Authentication configured.")

## 3. Dependencies & Codebase Retrieval

In [ ]:
print("📦 Installing dependencies...")
!pip install -q timm albumentations datasets huggingface_hub requests tqdm psutil opencv-python-headless
print("✅ Dependencies installed.\n")

print("📥 Cloning Repository...")
REPO_URL = "https://github.com/Vtheonly/AI_PROJECT_TAMER_Complete"
!git clone {REPO_URL}
print("✅ Codebase retrieved.")

## 4. Applying Critical Architectural Fixes
Overwriting `tokenizer.py`, `dataset.py`, and `encoder.py` directly into the cloned codebase. This applies ImageNet scaling, fixes the shape assertion crash, and correctly reshapes the 256x1024 feature map.

In [ ]:
%%writefile AI_PROJECT_TAMER_Complete/tamer_ocr/data/tokenizer.py
"""
LaTeX Tokenizer for Math OCR Training.
Builds a global vocabulary from all datasets. Special tokens are fixed at indices 0-3.
"""
import logging
from collections import Counter
from typing import List, Dict, Optional
import json
import os

logger = logging.getLogger("TAMER.Tokenizer")

class LaTeXTokenizer:
    PAD_TOKEN = '<pad>'
    SOS_TOKEN = '<sos>'
    EOS_TOKEN = '<eos>'
    UNK_TOKEN = '<unk>'
    SPECIAL_TOKENS = [PAD_TOKEN, SOS_TOKEN, EOS_TOKEN, UNK_TOKEN]

    def __init__(self):
        self.vocab: Dict[str, int] = {t: i for i, t in enumerate(self.SPECIAL_TOKENS)}
        self.reverse_vocab: Dict[int, str] = {i: t for t, i in self.vocab.items()}

    def tokenize(self, latex: str) -> List[str]:
        """Tokenize a LaTeX string. Crucially splits digits individually to prevent vocabulary explosion."""
        tokens = []
        i = 0
        latex = latex.strip()
        while i < len(latex):
            if latex[i].isspace():
                i += 1
                continue
            if latex[i] == '\\':
                j = i + 1
                while j < len(latex) and latex[j].isalpha():
                    j += 1
                if j == i + 1:
                    if j < len(latex):
                        tokens.append(latex[i:j+1])
                        j += 1
                    else:
                        tokens.append(latex[i])
                        j += 1
                else:
                    tokens.append(latex[i:j])
                i = j
            elif latex[i] in '{}()[]+-=_^&|<>':
                tokens.append(latex[i])
                i += 1
            elif latex[i].isdigit() or latex[i] == '.':  # FIX: Treat digits and decimals separately
                tokens.append(latex[i])
                i += 1
            else:
                tokens.append(latex[i])
                i += 1
        return tokens

    def build_from_corpus(self, corpus: List[str]):
        logger.info("Building vocabulary from corpus...")
        token_counts = Counter()
        for text in corpus:
            token_counts.update(self.tokenize(text))
        for token, _ in token_counts.most_common():
            if token not in self.vocab:
                self.vocab[token] = len(self.vocab)
        self.reverse_vocab = {i: t for t, i in self.vocab.items()}
        logger.info(f"Vocabulary built: {len(self.vocab)} total tokens.")

    def build_from_samples(self, samples: list):
        corpus = [s.get('latex', '') for s in samples if s.get('latex')]
        self.build_from_corpus(corpus)

    def encode(self, tokens: List[str]) -> List[int]:
        return [self.vocab.get(t, self.unk_id) for t in tokens]

    def decode(self, indices: List[int], skip_special: bool = True) -> str:
        res = []
        for idx in indices:
            t = self.reverse_vocab.get(idx, self.UNK_TOKEN)
            if skip_special and t in self.SPECIAL_TOKENS:
                continue
            if t == self.EOS_TOKEN:
                break
            res.append(t)
        return ' '.join(res)

    def save(self, path: str):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        with open(path, 'w', encoding='utf-8') as f:
            json.dump({'vocab': self.vocab}, f, ensure_ascii=False, indent=2)
        logger.info(f"Tokenizer saved to {path} ({len(self.vocab)} tokens)")

    def load(self, path: str):
        with open(path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        self.vocab = data['vocab']
        self.reverse_vocab = {int(i): t for t, i in self.vocab.items()}
        for token, expected_idx in zip(self.SPECIAL_TOKENS, range(4)):
            if self.vocab.get(token) != expected_idx:
                logger.warning(f"Special token {token} not at expected index {expected_idx}")
        logger.info(f"Tokenizer loaded from {path} ({len(self.vocab)} tokens)")

    @property
    def pad_id(self) -> int: return self.vocab[self.PAD_TOKEN]
    @property
    def sos_id(self) -> int: return self.vocab[self.SOS_TOKEN]
    @property
    def eos_id(self) -> int: return self.vocab[self.EOS_TOKEN]
    @property
    def unk_id(self) -> int: return self.vocab[self.UNK_TOKEN]
    def __len__(self) -> int: return len(self.vocab)


In [ ]:
%%writefile AI_PROJECT_TAMER_Complete/tamer_ocr/data/dataset.py
import torch
from torch.utils.data import Dataset
from torch.nn.utils.rnn import pad_sequence
import numpy as np
import cv2
import logging
from PIL import Image
from typing import List, Dict, Any, Union
from .tokenizer import LaTeXTokenizer

logger = logging.getLogger("TAMER.Dataset")

class MathDataset(Dataset):
    def __init__(self, samples: List[Dict[str, Any]], config, tokenizer: LaTeXTokenizer, transform=None):
        self.samples = samples
        self.config = config
        self.tokenizer = tokenizer
        self.transform = transform
        self._token_lengths = []
        for s in self.samples:
            tokens = self.tokenizer.tokenize(s.get('latex', ''))
            self._token_lengths.append(len(tokens))

    def __len__(self) -> int:
        return len(self.samples)

    def _process_image(self, img_source: Union[str, Image.Image]) -> torch.Tensor:
        target_h, target_w = self.config.img_height, self.config.img_width

        if isinstance(img_source, str):
            img = Image.open(img_source)
        elif isinstance(img_source, Image.Image):
            img = img_source
        else:
            raise ValueError(f"Unsupported image type: {type(img_source)}")

        img = img.convert('L')
        arr = np.array(img)

        def _blank_tensor():
            canvas = np.full((target_h, target_w), 255.0, dtype=np.float32)
            tensor = torch.from_numpy(canvas) / 255.0
            tensor = tensor.unsqueeze(0).expand(3, -1, -1)
            mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
            std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
            return (tensor - mean) / std

        if arr.size == 0 or arr.shape[0] == 0 or arr.shape[1] == 0:
            return _blank_tensor()

        h, w = arr.shape
        aspect = max(w, h) / max(min(w, h), 1)
        if aspect > self.config.max_aspect_ratio:
            return _blank_tensor()

        scale = target_h / h
        new_w = int(w * scale)

        if new_w > target_w:
            scale = target_w / w
            new_h = int(h * scale)
            arr = cv2.resize(arr, (target_w, new_h), interpolation=cv2.INTER_AREA)
            canvas = np.full((target_h, target_w), 255, dtype=np.uint8)
            y_offset = (target_h - new_h) // 2
            canvas[y_offset:y_offset+new_h, :] = arr
        else:
            arr = cv2.resize(arr, (new_w, target_h), interpolation=cv2.INTER_AREA)
            canvas = np.full((target_h, target_w), 255, dtype=np.uint8)
            canvas[:, :new_w] = arr

        if self.transform:
            canvas = self.transform(image=canvas)['image']

        # FIX: Standard ImageNet Normalization (No color inversion, expand to 3 channels)
        tensor = torch.from_numpy(canvas.astype(np.float32)) / 255.0
        tensor = tensor.unsqueeze(0).expand(3, -1, -1)
        mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        return (tensor - mean) / std

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        sample = self.samples[idx]
        img_source = sample.get('image') or sample.get('image_path')
        latex = sample.get('latex', '')
        dataset_name = sample.get('dataset_name', 'unknown')

        try:
            tensor = self._process_image(img_source)
        except Exception as e:
            logger.warning(f"Failed to load image (idx={idx}): {e}")
            canvas = np.full((self.config.img_height, self.config.img_width), 255.0, dtype=np.float32)
            tensor = torch.from_numpy(canvas) / 255.0
            tensor = tensor.unsqueeze(0).expand(3, -1, -1)
            mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
            std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
            tensor = (tensor - mean) / std
            latex = ""

        tokens = self.tokenizer.tokenize(latex)[:self.config.max_seq_len - 2]
        ids = [self.tokenizer.sos_id] + self.tokenizer.encode(tokens) + [self.tokenizer.eos_id]

        return {
            'image': tensor,
            'ids': torch.tensor(ids, dtype=torch.long),
            'length': len(ids),
            'latex': latex,
            'dataset_name': dataset_name,
        }

def get_collate_fn(pad_id: int):
    def collate_fn(batch):
        batch = [x for x in batch if x['length'] > 0]
        if not batch:
            return None
        images = torch.stack([x['image'] for x in batch])
        lengths = torch.tensor([x['length'] for x in batch])
        latices = [x['latex'] for x in batch]
        dataset_names = [x['dataset_name'] for x in batch]
        ids = pad_sequence([x['ids'] for x in batch], batch_first=True, padding_value=pad_id)
        return {'image': images, 'ids': ids, 'length': lengths, 'latex': latices, 'dataset_name': dataset_names}
    return collate_fn


In [ ]:
%%writefile AI_PROJECT_TAMER_Complete/tamer_ocr/models/encoder.py
"""
Swin-Base Encoder with Gradient Checkpointing.
"""
import torch
import torch.nn as nn
import timm
import logging

logger = logging.getLogger("TAMER.Encoder")

class SwinEncoder(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config

        # FIX: explicitly pass img_size to bypass strict shape assertions in timm
        # and allow dynamic size calculation for the 256x1024 grid.
        self.backbone = timm.create_model(
            config.encoder_name,
            pretrained=True,
            features_only=True,
            out_indices=(2,),
            img_size=(config.img_height, config.img_width)
        )

        self.backbone.set_grad_checkpointing(True)
        logger.info("Gradient checkpointing ENABLED on Swin backbone")

        dummy_input = torch.randn(1, 3, config.img_height, config.img_width)
        with torch.no_grad():
            dummy_out = self.backbone(dummy_input)[0]

        if dummy_out.dim() == 3:
            if dummy_out.shape[1] > dummy_out.shape[2]:
                self.format = "BLC"
                feature_dim = dummy_out.shape[2]
            else:
                self.format = "BCL"
                feature_dim = dummy_out.shape[1]
        else:
            if dummy_out.shape[1] > dummy_out.shape[-1]:
                self.format = "BCHW"
                feature_dim = dummy_out.shape[1]
            else:
                self.format = "BHWC"
                feature_dim = dummy_out.shape[-1]

        logger.info(f"Swin output format: {self.format}, feature_dim: {feature_dim}")
        
        self.proj = nn.Sequential(
            nn.Linear(feature_dim, config.d_model),
            nn.LayerNorm(config.d_model)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        features = self.backbone(x)[0]
        B = features.shape[0]

        # Convert everything to B, H, W, C
        if self.format == "BCHW":
            features = features.permute(0, 2, 3, 1)
        elif self.format == "BLC":
            # FIX: Do not assume square images! Calculate exact H and W.
            # Swin patch size is 4. Stage 2 is downsampled by an additional 4x (total 16x).
            H_feat = self.config.img_height // 16
            W_feat = self.config.img_width // 16
            features = features.view(B, H_feat, W_feat, -1)
        elif self.format == "BCL":
            features = features.permute(0, 2, 1)
            H_feat = self.config.img_height // 16
            W_feat = self.config.img_width // 16
            features = features.view(B, H_feat, W_feat, -1)

        features = self.proj(features)
        B, H, W, D = features.shape
        features = features.reshape(B, H * W, D)

        return features


## 5. System Path Injection & Configuration
Inject the codebase into the python path and configure the environment-specific directories dynamically.

In [ ]:
import sys

REPO_ROOT = os.path.join(WORK_DIR, 'AI_PROJECT_TAMER_Complete')
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print(f"🔗 System Path configured: {REPO_ROOT}")

from tamer_ocr.config import Config
from huggingface_hub import HfApi

# Initialize Configuration
config = Config()

# Dynamic Directory Mapping
config.data_dir = os.path.join(WORK_DIR, 'data')
config.output_dir = os.path.join(WORK_DIR, 'outputs')
config.checkpoint_dir = os.path.join(WORK_DIR, 'checkpoints')
config.log_dir = os.path.join(WORK_DIR, 'logs')

for d in [config.data_dir, config.output_dir, config.checkpoint_dir, config.log_dir]:
    os.makedirs(d, exist_ok=True)

# HuggingFace Hub Setup
hf_api = HfApi(token=os.environ['HF_TOKEN'])
hf_username = hf_api.whoami()['name']
config.hf_token = os.environ['HF_TOKEN']
config.hf_repo_id = f"{hf_username}/tamer-ocr-model"
config.hf_dataset_repo_id = f"{hf_username}/tamer-preprocessed"

# Training Hyperparameters
config.batch_size = 8
config.accumulation_steps = 4  # Effective batch = 32
config.num_epochs = 150
config.encoder_lr = 1e-5
config.decoder_lr = 1e-4
config.label_smoothing = 0.1
config.total_training_steps = 50000

# Checkpoint Safety
config.checkpoint_every_epochs = 3
config.keep_last_n_checkpoints = 3

print("✅ Configuration fully initialized and synchronized.")

## 6. Execute Main Pipeline (Strict Mode)
The `Trainer` class handles the strict order of operations:
1. Process all 4 datasets fully.
2. Push datasets to HuggingFace (to save state).
3. Initialize the model and DataLoaders.
4. Auto-resume from latest local or HF checkpoint if this is a restart.
5. Train.

In [ ]:
from tamer_ocr.core.trainer import Trainer

print("🚀 Launching TAMER OCR Pipeline...")
trainer = Trainer(config)
trainer.run()  # Fully self-contained execution loop

## 7. Final Output Synchronization & Hub Push
Ensures the absolute best checkpoint is correctly synced to the HuggingFace Hub.

In [ ]:
from tamer_ocr.utils.checkpoint import push_checkpoint_to_hf

best_model_path = os.path.join(config.checkpoint_dir, 'best.pt')
if os.path.exists(best_model_path):
    print(f"📤 Pushing best model to {config.hf_repo_id}...")
    push_checkpoint_to_hf(best_model_path, config, epoch=0, is_best=True)
    print("✅ Push complete.")
else:
    print("⚠️ No best.pt found. The model may not have finished the first evaluation cycle.")